<a href="https://colab.research.google.com/github/ced-sys/AI-N-ML/blob/main/Barbados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

In [4]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
import lightgbm as lgb
import joblib

In [5]:
!pip install ultralytics
from ultralytics import YOLO
import torch

  Using cached ultralytics-8.3.228-py3-none-any.whl.metadata (37 kB)
  Using cached ultralytics_thop-2.0.18-py3-none-any.whl.metadata (14 kB)
Using cached ultralytics-8.3.228-py3-none-any.whl (1.1 MB)
Using cached ultralytics_thop-2.0.18-py3-none-any.whl (28 kB)
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

In [7]:
class VideoFeatureExtractor:

  def __init__(self, model_name: str='yolov8n.pt'):
    print(f"Loading frozen YOLO model: {model_name}")
    self.model=YOLO(model_name)
    self.model.model.eval()

    for param in self.model.model.parameters():
      param.requires_grad=False

      self.vehicle_classes={
          2:'car',
          3:'motorcycle',
          5:'bus',
          7:'truck'
      }

  def extract_frame_features(self, frame: np.ndarray) -> Dict:
    with torch.no_grad():
      results=self.model(frame, verbose=False)[0]

    boxes=results.boxes

    features={
        'total_vehicles':0,
        'cars':0,
        'motorcycles':0,
        'buses':0,
        'trucks':0,
        'avg_vehicle_area':0,
        'vehicle_density':0,
        'avg_confidence':0,
        'vehicle_positions_x':[],
        'vehicle_positions_y':[],
        'vehicle_areas':[]
    }

    if len(boxes)==0:
      return features

    vehicle_detections=[]
    for box in boxes:
      cls_id=int(box.cls[0])
      conf=float(box.conf[0])

      if cls_id in self.vehicle_classes and conf >0.3:
        x1, y1, x2, y2=box.xyxy[0].cpu().numpy()
        area=(x2-x1)*(y2-y1)
        center_x=(x1+x2)/2
        center_y=(y1+y2)/2

        vehicle_detections.append({
            'type':self.vehicle_classes[cls_id],
            'conf':conf,
            'area':area,
            "center_x":center_x,
            'center_y':center_y
        })

        features[self.vehicle_classes[cls_id]+'s']+=1
        features['vehicle_areas'].append(area)
        features['vehicle_positions_x'].append(center_x)
        features['vehicle_positions_y'].append(center_y)


    features['total_vehicles']=len(vehicle_detections)

    if features['total_vehicles']>0:
      features['avg_vehicle_area']=np.mean(features['vehicle_areas'])
      features['avg_confidence']=np.mean([d['conf']for d in vehicle_detections])

      frame_area=frame.shape[0]*frame.shape[1]
      features['vehicle_density']=features['total_vehicles']/ (frame_area/1e6)

      features['spatial_std_x']=np.std(features['vehicle_positions_x'])
      featured['spatial_std_y']=np.std(features['vehicle_positions_y'])

    return features

  def extract_video_features(self, video_path: str, sample_rate: int=5)-> Dict:
    cap=cv2.VideoCapture(video_path)

    if not cap.isOpened():
      print(f"Error opening video: {video_path}")
      return self._get_empty_features()

    frame_features=[]
    frame_count=0

    while True:
      ret, frame=cap.read()
      if not ret:
        break

      if frame_count % sample_rate==0:
        features=self.extract_frame_features(frame)
        frame_features.append(features)

      frame_count+=1

    cap.release()

    if not frame_features:
      return self._get_empty_features()

    return self._aggregate_frame_features(frame_features, frame_count)

  def _aggregate_frame_features(self, frame_features: List[Dict], total_frames: int)-> Dict:
    agg_features={}

    vehicle_Counts=[f['total_vehicles']for f in frame_features]
    agg_features['mean_vehicles']=np.mean(vehicle_counts)
    agg_features['max_vehicles']=np.max(vehicle_counts)
    agg_features['min_vehicles']=np.min(vehicle_counts)
    agg_features['std_vehicles']=np.std(vehicle_counts)
    agg_features['median_vehicles']=np.median(vehicle_counts)

    for vtype in ['cars', 'motorcycles', 'buses', 'trucks']:
      counts=[f[vtype] for f in frame_features]
      agg_features[f'mean_{vtype}']=np.mean(counts)
      agg_features[f'max_{vtype}']=np.max(counts)

    densities=[f['vehicle_density'] for f in frame_features]
    agg_features['mean_density']=np.mean(densities)
    agg_features['max_density']=np.max(densities)

    areas=[f['avg_vehicle_area'] for f in frame_features if f['avg_vehicle_area']>0]
    agg_features['mean_vehicle_area']=np.mean(areas) if areas else 0

    if len(vehicle_counts)>1:
      flow=np.diff(vehicle_counts)
      agg_features['mean_flow_change']=np.mean(flow)
      agg_features['flow_volatility']=np.std(flow)
      agg_features['positive_flow_ratio']=np.sum(flow>0)/ len(flow)
    else:
      agg_features['mean_flow_change']=0
      agg_features['flow_volatility']=0
      agg_features['postive_flow_ration']=0

    agg_features['congestion_score']=(
        agg_features['mean_vehicles']*0.4+
        agg_features['max_vehicles']*0.3+
        agg_features['mean_density']*0.3
    )


    agg_features['total_frames']=total_frames
    agg_features['sampled_frames']=len(frame_features)

    return agg_features

  def _get_empty_features(self)-> Dict:
    return{
        'mean_vehicles':0, 'max_vehicles':0, 'min_vehicles':0,
        'std_vehicles':0, 'median_vehicles':0,
        'mean_cars':0, 'max_cars':0,
        'mean_motorcycles':0, 'max_motorcycles':0,
        'mean_buses':0, 'max_buses':0,
        'mean_trucks':0, 'max_trucks':0,
        'mean_density':0, 'max_density':0,
        'mean_vehicle_area':0,
        'mean_flow_change':0, 'flow_volatility':0,
        'positive_flow_ratio':0,
        'congestion_score':0,
        'total_frames':0, 'sampled_frames':0
    }




In [8]:
class TemporalFeatureEngineer:
  @staticmethod
  def create_lag_features(df: pd.DataFrame, feature_cols: List[str],
                          lags: List[int], group_col: str='camera')-> pd.DataFrame:

    df=df.copy()

    for col in feature_cols:
      for lag in lags:
        df[f'{col}_lag{lag}']=df.groupby(group_col)[col].shift(lag)

    return df

  @staticmethod
  def create_rolling_features(df: pd.DataFrame, feature_cols: List[str],
                              windows: List[int], group_col: str='camera')-> pd.DataFrame:
    df=df.copy()

    for col in feature_cols:
      for window in windows:
        df[f'{col}_rolling{window}_mean']=(
            df.groupby(group_col)[col]
            .rolling(window=window, min_periods=1)
            .mean()
            .reset_index(0, drop=True)
        )

        df[f'{col}_rolling{window}_std']=(
            df.groupby(group_col)[col]
            .rolling(window=window, min_periods=1)
            .std()
            .reset_index(0, drop=True)
        )
        df[f'{col}_rolling{window}_max']=(
            df.groupby(group_col)[col]
            .rolling(window=window, min_periods=1)
            .max()
            .reset_index(0, drop=True)
        )

    return df

  @staticmethod
  def create_time_features(df: pd.DataFrame, tie_col: str='time_segment')-> pd.DataFrame:
    df=df.copy()

    df['segment_num']=df[time_col].str.extract(r'(\d+)').astype(int)

    df['segment_sin']=np.sin(2*np.pi*df['segment_num']/df['segment_num'].max())
    df['segment_cos']=np.sin(2*np.pi*df['segment_num']/df['segment_num'].max())

    return df